In [31]:
import numpy as np
import pandas as pd
import re
import string
import pickle

In [10]:
txt = "great product, I loved it!"

In [11]:
def remove_punctuations(text):
    for punctuation in string.punctuation:
        text = text.replace(punctuation, "")
    return text 

In [12]:
with open('../static/model/corpora/stopwords/english', 'r') as file:
    sw = file.read().splitlines()

In [13]:
from nltk.stem import PorterStemmer
ps = PorterStemmer()

In [21]:
def preprocess_text(text):
    data = pd.DataFrame([text], columns=["tweet"])
    data["tweet"] = data["tweet"].apply(lambda x: " ".join(x.lower() for x in x.split()))
    data["tweet"] = data["tweet"].apply(lambda x: " ".join(re.sub(r'^https?:\/\/.*[\r\n]*', '', x, flags=re.MULTILINE) for x in x.split()))
    data["tweet"] = data["tweet"].apply(remove_punctuations)
    data["tweet"] = data["tweet"].str.replace('\d+', '', regex=True)
    data["tweet"] = data["tweet"].apply(lambda x: " ".join(x for x in x.split() if x not in sw))
    data["tweet"] = data["tweet"].apply(lambda x: " ".join(ps.stem(x) for x in x.split()))
    return data["tweet"]

<>:6: SyntaxWarning: invalid escape sequence '\d'
<>:6: SyntaxWarning: invalid escape sequence '\d'
C:\Users\USER\AppData\Local\Temp\ipykernel_21036\3305111960.py:6: SyntaxWarning: invalid escape sequence '\d'
  data["tweet"] = data["tweet"].str.replace('\d+', '', regex=True)


In [22]:
preprocess_txt = preprocess_text(txt)


In [25]:
vocab = pd.read_csv('../static/model/vocab.txt', header=None)
tokens = vocab[0].tolist()

In [28]:
def vectorize(ds, vocab):
    vectorized_ds = []
    for sentence in ds:
        vectorized_sentence = np.zeros(len(vocab))
        for i in range(len(vocab)):
            if vocab[i] in sentence:
                vectorized_sentence[i] = 1
        vectorized_ds.append(vectorized_sentence)
    vectorized_list = np.asarray(vectorized_ds, dtype=np.float32)
    return vectorized_list

In [38]:
with open('../static/model/sentiment_model.pkl', 'rb') as file:
    model = pickle.load(file)

In [39]:
def get_prediction(vectorized_text):
    prediction = model.predict(vectorized_text)
    if prediction == 1:
        return 'negetive'
    else:
        return 'positive'

In [29]:
vectorized_text = vectorize(preprocess_txt, tokens)

In [34]:
model.predict(vectorized_text)

array([0])

In [40]:
txt2 = 'bad product, I hated it!'
preprocessed_text2 = preprocess_text(txt2)
vectorized_text2 = vectorize(preprocessed_text2, tokens)
prediction2 = get_prediction(vectorized_text2)
prediction2

'negetive'